<a href="https://colab.research.google.com/github/Jake0925/LLM/blob/main/RAG_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 최신 패키지 설치
!pip install -U langchain langchain-community faiss-cpu tiktoken sentence-transformers langchain-huggingface

In [2]:
try:
  from langchain_community.vectorstores import FAISS
  from langchain_text_splitters import CharacterTextSplitter
  from langchain_huggingface import HuggingFaceEmbeddings
  from langchain_community.vectorstores import FAISS
  print("라이브러리 로드 성공!")
except ModuleNotFoundError:
    print("오류: 'langchain_huggingface'를 찾을 수 없습니다. 상단 메뉴 [런타임] -> [세션 다시 시작] 후 다시 실행해 주세요.")

라이브러리 로드 성공!


In [3]:
# 4. 테스트용 문서
documents = [
    "우리 회사의 휴가 정책은 연 15일입니다.",
    "환불은 구매 후 7일 이내에 가능합니다.",
    "고객센터 운영시간은 평일 9시부터 18시까지입니다."
]

In [4]:
# 5. 텍스트 분할
text_splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=10)
texts = text_splitter.create_documents(documents)

In [5]:
# 무료로 사용할 수 있는 한국어 지원 모델을 로드합니다.
model_name = "jhgan/ko-sroberta-multitask"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

try:
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )

    # 기존의 texts(Document 객체 리스트)를 사용하여 벡터 DB 생성
    vectorstore = FAISS.from_documents(texts, embeddings)
    print("HuggingFace 모델로 벡터 데이터베이스 생성 완료!")
except Exception as e:
    print(f"오류 발생: {e}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFace 모델로 벡터 데이터베이스 생성 완료!


# Retrieval 확인 (중요🔥)

In [6]:
query = "우리 회사의 휴가 정책에 대해 알려줘."

# 1. 유사도 검색 수행 (k=2는 가장 유사한 문서 2개를 가져오라는 의미입니다)
retrieved_docs = vectorstore.similarity_search(query, k=2)

print(f"질문: {query}\n")
print("--- 검색된 결과 ---")
for i, doc in enumerate(retrieved_docs):
    print(f"[{i+1}] {doc.page_content}")

질문: 우리 회사의 휴가 정책에 대해 알려줘.

--- 검색된 결과 ---
[1] 우리 회사의 휴가 정책은 연 15일입니다.
[2] 고객센터 운영시간은 평일 9시부터 18시까지입니다.


# 간단 자동 평가 (정확도 체크)

In [7]:
def simple_evaluation(query, retrieved_docs, ground_truth_keywords):
    # 검색된 모든 문서의 내용을 하나의 텍스트로 합칩니다.
    combined_content = " ".join([doc.page_content for doc in retrieved_docs])

    print(f"평가 질문: {query}")
    print(f"기대 키워드: {ground_truth_keywords}")

    results = {}
    for keyword in ground_truth_keywords:
        is_found = keyword in combined_content
        results[keyword] = "✅ 통과" if is_found else "❌ 미흡"

    return results

# 평가 실행
# '휴가' 정책 질문이므로 '15일'이라는 핵심 키워드가 포함되었는지 확인합니다.
test_keywords = ["15일", "휴가"]
eval_results = simple_evaluation(query, retrieved_docs, test_keywords)

print("\n--- 평가 결과 ---")
for kw, res in eval_results.items():
    print(f"{kw}: {res}")

평가 질문: 우리 회사의 휴가 정책에 대해 알려줘.
기대 키워드: ['15일', '휴가']

--- 평가 결과 ---
15일: ✅ 통과
휴가: ✅ 통과


# 실패 케이스 테스트

In [8]:
# 1. 문서에 없는 내용을 질문 (실패 케이스 테스트)
fail_query = "애플 페이 결제가 가능한가요?"
fail_retrieved_docs = vectorstore.similarity_search(fail_query, k=2)

# 2. '결제'나 '애플' 같은 키워드가 포함될 것으로 기대하지만, 실제 문서에는 없습니다.
fail_keywords = ["애플", "결제"]
fail_eval_results = simple_evaluation(fail_query, fail_retrieved_docs, fail_keywords)

print("\n--- [실패 케이스] 검색된 결과 ---")
for i, doc in enumerate(fail_retrieved_docs):
    print(f"[{i+1}] {doc.page_content}")

print("\n--- [실패 케이스] 평가 결과 ---")
for kw, res in fail_eval_results.items():
    print(f"{kw}: {res}")

평가 질문: 애플 페이 결제가 가능한가요?
기대 키워드: ['애플', '결제']

--- [실패 케이스] 검색된 결과 ---
[1] 환불은 구매 후 7일 이내에 가능합니다.
[2] 고객센터 운영시간은 평일 9시부터 18시까지입니다.

--- [실패 케이스] 평가 결과 ---
애플: ❌ 미흡
결제: ❌ 미흡
